In [70]:
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import time
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import numpy as np

import random

def set_seed(seed):
    """Sets the seed for reproducibility."""
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU.
        
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python's random module.
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False

set_seed(1000)

In [68]:
# Import pre-split datasets
df_train = pd.read_csv('./data/bbq-train.csv')
df_test = pd.read_csv('./data/bbq-test.csv')

## Text processing

In [71]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text):
    # Some reviews are only numbers (Why? I don't know). Review should always be treated as a string
    text = str(text)
    
    # This appears in some reviews
    text = text.replace('(Translated by Google)', ' ')
    text = text.replace('\n', ' ')

    # Convert to lowercase
    text = text.lower()
    
    # Remove non letter or number entries
    # BERT does OK with numbers is seems
    text = re.sub(r'[^\w\s]', '', text)
    
    return text

# Process the text for better classification
df_train['text'] = df_train['text'].apply(preprocess_text)
df_test['text'] = df_test['text'].apply(preprocess_text)

In [72]:
# Turn ratings into +/-/neutral and do a bit of light processing to the text
# Careful to only run this once

def pnn(n):
    if n in {1, 2}: return 0
    elif n in {3}: return 1
    else: return 2

def minus1(n):
    return n-1
    
df_train['pnn'] = df_train['rating'].apply(pnn)
df_test['pnn'] = df_test['rating'].apply(pnn)


# df_train['rating'] = df_train['rating'].apply(minus1)
# df_test['rating'] = df_test['rating'].apply(minus1)

In [73]:
print('Train set sample')
print(df_train.sample(8))
print()

print('Test set sample')
print(df_test.sample(8))

Train set sample
        rating                                               text  pnn
13816        5  brisket and pasta salad are awesome the best o...    2
77405        5  excellent korean food and service had pot stic...    2
58074        5       their twist ice cream cone is the phenomenal    2
26985        5                       nice night out with my hubby    2
131277       3    great food but bathroom was filthy and freezing    1
39949        5  omg must order the wings im addicted cant beli...    2
126221       3  love the polish girlboy but didnt like the mac...    1
21877        5  this is my goto place for barbecue from now on...    2

Test set sample
       rating                                               text  pnn
27369       3  it is exactly alright  in indy its the best ba...    1
27129       4                very good food and friendly service    2
14326       5                                            awesome    2
8401        5                       great food 

Instantiate the bert model. Tried both base and large, no real difference in acc

## Create BERT instance
Skip this step if continuing training

In [74]:
# from transformers import BertTokenizer, BertModel, BertForSequenceClassification
from transformers import AutoModelForSequenceClassification, TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig, get_linear_schedule_with_warmup

model_name = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# Model stats
#print(model)
print('Device: ', device)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device:  cuda


## Test the output of an untrained model

In [78]:
# Simple test cycle of the untrained model
def predict_sentiment(text):
    inputs = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    inputs = {'input_ids':input_ids, 'attention_mask':attention_mask}
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        
    return predicted_class, probabilities.tolist()

# Enter your text here
review = 'Sucks!'
label, probs = predict_sentiment(review)
meaning = ['negative', 'neutral/average', 'good']

print(f'Review: {review}')
print(f'Predicted rating: {label} ({meaning[label]})')
print(f'Probabilities: 0 ({probs[0][0]*100:.2f}%) | 1 ({probs[0][1]*100:.2f}%) | 2 ({probs[0][2]*100:.2f}%)')

Review: Sucks!
Predicted rating: 0 (negative)
Probabilities: 0 (84.86%) | 1 (12.78%) | 2 (2.35%)


## Script for autogenerating plot of accuracies and runtime


In [79]:
import datetime
import matplotlib.pyplot as plt

def plot_lists_with_colors(list1, list2, label1="List 1", label2="List 2", color1="blue", color2="red"):
    # Ensure lists have equal length (or adjust as needed)
    min_len = min(len(list1), len(list2))
    x_values = np.arange(min_len)

    # Create the plot
    plt.figure(figsize=(10, 6))  # Adjust figure size if needed

    # Plot List 1
    plt.plot(x_values, list1[:min_len], color=color1, label=label1, linestyle='none', marker='o')
    plt.plot(x_values, list2[:min_len], color=color2, label=label2, linestyle='none', marker='o')
    current_time = datetime.datetime.now()
    
    # Add Labels and Title
    plt.xlabel("Epoch") 
    plt.ylabel("Accuracy (%)")
    plt.title(f"Bert - Midwest data ({current_time.strftime("%Y-%m-%d %H:%m")})")

    plt.legend()
    plt.grid(True)

    file_name = f'./sshots/model-{current_time.strftime("%Y-%m-%d--%H%m")}.png'
    plt.savefig(file_name)
    
    # Show the Plot
    plt.show()

def runtime(start_time):
    end_time = time.time()
    s = end_time - start_time

    h = s // 3600
    s = s - 3600*h

    m = s // 60
    s = s - 60*m

    print(f'Total runtime: {int(h)} hr {int(m)} min {s:.1f} sec')
    return 

## Training the BERT model begins here

In [80]:
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split


# Create dataset from data
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length #store max length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            text,
            padding="max_length", #important
            truncation=True, #important
            max_length=self.max_length, #important
            return_tensors="pt")
        
        input_ids = inputs['input_ids'].flatten()
        attention_mask = inputs['attention_mask'].flatten()
        label_tensor = torch.tensor(label)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_tensor
        }

In [81]:
from sklearn.utils import class_weight

# Ratings are not evenly distributed, this creates class weights
def calculate_class_weights(labels):
    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)

In [82]:
# Export the text and ratings to a list, this part is necessary otherwise pandas keeps the index
X_train = df_train['text'].values.tolist()
y_train = df_train['pnn'].values.tolist()
X_test = df_test['text'].values.tolist()
y_test = df_test['pnn'].values.tolist()

# Batch size set as a power of 2
# monitor RAM use in the terminal with "watch -n5 nvidia-smi"
batch_size = 64
max_length = 128

## How interact the scheduler?
learning_rate = 2e-5
weight_decay = .01

## FREEZE the base BERT parameters and only train the classification layer
for param in model.roberta.parameters():
    param.requires_grad = False

train_dataset = SentimentDataset(X_train, y_train, tokenizer, max_length=max_length)
test_dataset = SentimentDataset(X_test, y_test, tokenizer, max_length=max_length)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

# Adjust learning rate and weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Calculate class weights only after the split
class_weights = calculate_class_weights(y_train).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Class weights:', class_weights)

Class weights: tensor([3.6109, 4.1570, 0.4028], device='cuda:0')


In [107]:
def test_cycle(model, dataloader):
    # Put in evaluation mode
    print('Testing...', end='\r')
    model.eval()
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    ## Accuracy and F1 used to determine early exit
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return acc, f1

In [108]:
def train_roberta_early_stopping(model, train_dataloader, test_dataloader, optimizer, epochs=100, patience=5, device='cuda'):
    
    start_time = time.time()
    
    # Keep track of some data
    accs = [[], []] #0 = Train, 1 = Test
    losses = []
    f1s = []
    
    num_training_steps = len(train_dataloader) * epochs

    # Learning rate scheduler
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

    # Set best value as low as possible to start
    best_val_f1 = -np.inf
    patience_counter = 0

    for epoch in range(epochs):
        
        model.train()
    
        train_loss = 0
        all_preds, all_labels = [], []
    
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}"):
            optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    
            ## used for calculating accuracy
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
            ## Training
            loss = outputs.loss
            train_loss += loss.item()
            loss.backward()
            optimizer.step()
            scheduler.step()

        # Keep track of learning acc and loss
        acc = accuracy_score(all_labels, all_preds)
        trin_loss = train_loss / len(train_dataloader)
        
        accs[0].append(100*acc)
        losses.append(loss)
        
        # Test cycle
        acc, val_f1 = test_cycle(model=model, dataloader=test_dataloader)
        
        accs[1].append(100*acc)
        f1s.append(val_f1)

        # Keep track of progress
        print(f'Train loss: {loss:.5f}, acc {accs[0][-1]:.2f}% | Test acc: {accs[1][-1]:.2f}%, f1 = {val_f1:.5f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), './saved-models/roberta-bbq.pth') #save best model
            
        else:
            patience_counter += 1
            if patience_counter >= patience:
                
                print('Early stopping triggered')
                plot_lists_with_colors(accs[0], accs[1], label1="Train accuracy", label2="Test accuracy")
                break #exit training loop.
    
    runtime(start_time)
    model.load_state_dict(torch.load('./saved-models/roberta-bbq.pth')) #load best model   
    return model

In [ ]:
# Train the model here
model = train_roberta_early_stopping(model=model, 
                                     train_dataloader=train_dataloader, 
                                     test_dataloader=test_dataloader, 
                                     optimizer=optimizer, 
                                     epochs=100, 
                                     patience=5,
                                     device=device)

Trianing on device (cuda) with 100 epochs


Epoch 1: 100%|██████████████████████████████| 2301/2301 [06:01<00:00,  6.37it/s]


Train loss: 0.15397, acc 89.89% | Test acc: 90.31%, f1 = 0.89474


Epoch 2: 100%|██████████████████████████████| 2301/2301 [06:01<00:00,  6.37it/s]


Train loss: 0.37158, acc 89.89% | Test acc: 90.37%, f1 = 0.89370


Epoch 3: 100%|██████████████████████████████| 2301/2301 [06:03<00:00,  6.33it/s]


## Suggestions

 Set aside cross-validation set - 

 Learning rates: 5e-5, 4e-5, 3e-5, and 2e-5, .0001. Learning rate scheduler

 Investigate BERT classifier more

 Try different roberta models